# r2 · Crisis exclusion, GARCH and Elastic Net vs `hist_rw`

Do the e6 results for GARCH and Elastic Net, compared with the historical-volatility benchmark
`hist_rw`, depend on the GFC or COVID months? We drop those months from the evaluation only:
nothing is refitted, and the saved forecasts are rescored on fewer months.

Samples (`crisis_config.py`): full, ex-GFC, ex-COVID, ex-both. The windows are the NBER recessions, matched on the month being forecast (target_date). Because the first forecast is for March 2008, the GFC part of the evaluation sample is March 2008 to June 2009. The initial training window already contains the first three GFC months (December 2007 to February 2008), and later expanding windows keep all GFC observations. Crisis exclusion changes only which months are scored and no model is refitted.

**Tests**, the same as e6: QLIKE uses DM for every model. Log-RV MSE uses DM for every model, plus
Clark-West for the Elastic Net variants, which nest `hist_rw`. Pooled differentials are averaged
across ETFs each month and tested with Newey-West (Andrews bandwidth, recomputed on each sample).
P-values are one-sided, H1: the model beats `hist_rw`. Dropping COVID leaves a three-month gap,
which the lags ignore.

Output: `models/forecasting/robustness/results/r2_crisis_naive_vs_garch_en_evaluation.csv`. `a` is the model and `b` is
`hist_rw`. `loss_ratio` is loss(a) / loss(b), and `pct_change = (loss_ratio - 1) * 100`.

In [ ]:
import sys
from pathlib import Path

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))
sys.path.insert(0, str(_checkout / "models" / "forecasting" / "robustness"))
from utils.paths import MODELS_DIR, FORECAST_DIR, FORECAST_RESULTS_DIR
from utils.portfolio_utils import FORECAST_FILES

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

from research_config import SELECTED, BENCHMARK, TICKERS
from crisis_config import SAMPLES, exclude

RESULTS_DIR = MODELS_DIR / "forecasting" / "robustness" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Families whose variants nest hist_rw: each Elastic Net includes log_rv_1m.
NESTED_FAMILIES = {"Elastic Net"}

CRISIS_SELECTED = {family: variants for family, variants in SELECTED.items()
                   if family != "XGBoost"}

frames = []
for fam in CRISIS_SELECTED:
    f = pd.read_csv(FORECAST_DIR / FORECAST_FILES[fam], parse_dates=["date"]).assign(family=fam)
    # All forecasts target the next month end (checked in e6).
    frames.append(f.assign(target_date=f["date"] + pd.offsets.MonthEnd(1)))
df = pd.concat(frames, ignore_index=True)

selected = [m for ms in CRISIS_SELECTED.values() for m in ms]
models = [m for m in selected if m in set(df["model"])]
df = df[df["model"].isin(models) & df["etf"].isin(TICKERS)].copy()
if df.duplicated(["model", "etf", "date"]).any():
    raise ValueError("Duplicate model/ETF/forecast-month rows.")

family = {m: fam for fam, ms in SELECTED.items() for m in ms}
nested = {m: family[m] in NESTED_FAMILIES for m in models}
# Same pair ids as e6, so the full sample can be checked against e6_evaluation.csv.
pairs = [(f"E6.{i}", m, BENCHMARK, family[m])
         for i, m in enumerate([m for variants in SELECTED.values() for m in variants if m != BENCHMARK], 1)
         if m in models]
print(f"{len(models)} of {len(selected)} models available; {len(pairs)} comparisons with {BENCHMARK}")

In [ ]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Sample

Common sample: the (ETF, month) rows with forecasts from the historical benchmark, GARCH
and Elastic Net models. Each sample below removes the specified crisis months.

In [ ]:
d = df[df.groupby(["etf", "date"])["model"].transform("nunique") == len(models)].copy()
d = d.sort_values(["date", "etf", "model"]).reset_index(drop=True)
# Keep the pooled monthly average equally weighted across the five ETFs.
if (d.groupby("date")["etf"].nunique() != len(TICKERS)).any():
    raise ValueError("The common sample must contain all five ETFs each month.")

d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])

months = {s: exclude(d, s)["target_date"].nunique() for s in SAMPLES}
pd.Series(months, name="forecast months").to_frame().T

## Tests by sample

In [ ]:
LOSSES = ["QLIKE", "MSE_log", "MSE_level"]


def evaluate(sub, sample):
    # Same losses and tests as e6, on the months kept in `sub`.
    per_etf = sub.groupby(["model", "etf"])[LOSSES].mean().reset_index()
    pooled = sub.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
    loss_idx = pd.concat([per_etf, pooled]).set_index(["model", "etf"])
    n_months = sub["date"].nunique()
    rows = []
    for pair_id, a, b, fam in pairs:
        A = sub[sub["model"] == a].set_index(["date", "etf"]).sort_index()
        B = sub[sub["model"] == b].set_index(["date", "etf"]).sort_index()
        mse_diff = A["MSE_log"] - B["MSE_log"]
        # DM differentials: negative favours a. Clark-West: positive favours a.
        comparisons = [("QLIKE", "DM", A["QLIKE"] - B["QLIKE"]), ("MSE_log", "DM", mse_diff)]
        if nested[a]:
            comparisons.append(("MSE_log", "Clark-West", -mse_diff + (B["fc_log"] - A["fc_log"]) ** 2))

        for etf in ["pooled"] + TICKERS:
            for loss, test, diff in comparisons:
                series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
                t = -hac_t(series) if test == "DM" else hac_t(series)
                rows.append({
                    "sample": sample, "pair": pair_id, "a": a, "b": b, "family": fam, "ETF": etf,
                    "loss": loss, "test": test,
                    "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                    "gain_t": t, "p_value": float(norm.sf(t)), "n_months": n_months,
                })
    return pd.DataFrame(rows)


results = pd.concat([evaluate(exclude(d, s), s) for s in SAMPLES], ignore_index=True)
results["pct_change"] = (results["loss_ratio"] - 1) * 100   # negative: a has the lower loss
results.to_csv(RESULTS_DIR / "r2_crisis_naive_vs_garch_en_evaluation.csv", index=False)

# The full sample must reproduce e6. A mismatch usually means a forecast file was regenerated
# after e6 last ran: rerun e6 first.
e6 = pd.read_csv(FORECAST_RESULTS_DIR / "e6_evaluation.csv")
e6 = e6[e6["a"].isin(models)].copy()
check = results[results["sample"] == "full"].merge(
    e6, on=["pair", "ETF", "loss", "test"], suffixes=("", "_e6"))
assert len(check) == len(e6), "e6_evaluation.csv has different pairs; rerun e6_naive_vs_all_models.ipynb"
for col in ["loss_ratio", "gain_t", "p_value"]:
    bad = check.loc[~np.isclose(check[col], check[col + "_e6"]), "a"].unique()
    assert len(bad) == 0, f"{col} differs from e6_evaluation.csv for {list(bad)}; rerun e6_naive_vs_all_models.ipynb"
print("full sample matches e6_evaluation.csv")

## Pooled results

Each cell is `loss_ratio (p-value)`; a ratio below 1 means the model has a lower loss than
`hist_rw`. Clark-West is shown for the Elastic Net variants only, since only they nest `hist_rw`.

In [ ]:
def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sample(loss, test, etf="pooled"):
    t = results[results["ETF"].eq(etf) & results["loss"].eq(loss) & results["test"].eq(test)].copy()
    t["cell"] = [f"{r:.3f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="sample", values="cell")
    return table.loc[t["row"].drop_duplicates(), list(SAMPLES)].rename_axis(None)


for loss, test in [("MSE_log", "DM"), ("MSE_log", "Clark-West"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sample(loss, test))

## Percentage changes, pooled

Percentage change in mean loss for each model relative to `hist_rw`: `100 × (loss_ratio - 1)`.
Negative values mean lower loss than the benchmark. The ratio does not depend on the test, so
the DM rows are used.

In [ ]:
for loss in ["MSE_log", "QLIKE"]:
    pct = (
        results[results["ETF"].eq("pooled") & results["loss"].eq(loss) & results["test"].eq("DM")]
        .assign(row=lambda t: t["pair"] + ": " + t["a"])
        .pivot(index="row", columns="sample", values="pct_change")
    )
    pct = pct.loc[[f"{pair_id}: {a}" for pair_id, a, _, _ in pairs], list(SAMPLES)]
    print(f"{'Log MSE' if loss == 'MSE_log' else loss}: % change relative to {BENCHMARK}")
    display(pct.rename_axis(None, axis=0).rename_axis(None, axis=1).map(lambda v: f"{v:+.2f}%"))